# 02 — Baselines: FinBERT + VADER
Fast runs on both datasets. Results saved to `results/summary/baselines.csv`.

In [ ]:
import sys, os, json
# Colab: uncomment
# PROJECT_DIR = '/content/drive/MyDrive/finllama-sentiment'
# sys.path.insert(0, PROJECT_DIR); os.chdir(PROJECT_DIR)

import pandas as pd
from src.data_loader import load_fpb, load_fiqa
from src.evaluation import compute_metrics
from src.utils import load_config, set_seed

cfg = load_config()
set_seed(cfg['seed'])
os.makedirs(cfg['paths']['predictions_dir'], exist_ok=True)
os.makedirs(cfg['paths']['summary_dir'], exist_ok=True)

In [ ]:
_, fpb_test = load_fpb(
    config=cfg['datasets']['fpb']['config'],
    test_fraction=cfg['datasets']['fpb']['test_fraction'],
    seed=cfg['seed'],
)
fiqa_test = load_fiqa(neutral_band=cfg['datasets']['fiqa']['neutral_band'])
datasets = {'FPB': fpb_test, 'FiQA': fiqa_test}
print({k: len(v) for k, v in datasets.items()})

## VADER

In [ ]:
import time
from src.models.vader_runner import VADERRunner

vader = VADERRunner()
rows = []

for ds_name, samples in datasets.items():
    texts = [s['text'] for s in samples]
    t0 = time.perf_counter()
    labels = vader.predict(texts)
    runtime_s = time.perf_counter() - t0

    preds = [
        {'id': s['id'], 'pred_label': lbl, 'raw_output': '', 'parse_ok': True, 'latency_ms': 0.0}
        for s, lbl in zip(samples, labels)
    ]

    run_id = f'vader__{ds_name}__seed{cfg["seed"]}'
    run_dir = os.path.join(cfg['paths']['predictions_dir'], run_id)
    os.makedirs(run_dir, exist_ok=True)
    with open(os.path.join(run_dir, 'predictions.jsonl'), 'w') as f:
        for p in preds: f.write(json.dumps(p) + '\n')

    m = compute_metrics(samples, preds)
    rows.append({'model': 'VADER', 'dataset': ds_name, 'template': '-', 'shots': 0,
                 'seed': cfg['seed'], **{k: m[k] for k in ('accuracy','f1_macro','f1_weighted','coverage','n_samples')},
                 'runtime_s': round(runtime_s, 2)})
    print(f'VADER {ds_name}: acc={m["accuracy"]:.3f}  f1_macro={m["f1_macro"]:.3f}')

## FinBERT

In [ ]:
from src.models.finbert_runner import FinBERTRunner

# device=0 for GPU, -1 for CPU
finbert = FinBERTRunner(hf_id=cfg['models']['finbert']['hf_id'], device=0)

for ds_name, samples in datasets.items():
    texts = [s['text'] for s in samples]
    t0 = time.perf_counter()
    labels = finbert.predict(texts)
    runtime_s = time.perf_counter() - t0

    preds = [
        {'id': s['id'], 'pred_label': lbl, 'raw_output': '', 'parse_ok': True, 'latency_ms': 0.0}
        for s, lbl in zip(samples, labels)
    ]

    run_id = f'finbert__{ds_name}__seed{cfg["seed"]}'
    run_dir = os.path.join(cfg['paths']['predictions_dir'], run_id)
    os.makedirs(run_dir, exist_ok=True)
    with open(os.path.join(run_dir, 'predictions.jsonl'), 'w') as f:
        for p in preds: f.write(json.dumps(p) + '\n')

    m = compute_metrics(samples, preds)
    rows.append({'model': 'FinBERT', 'dataset': ds_name, 'template': '-', 'shots': 0,
                 'seed': cfg['seed'], **{k: m[k] for k in ('accuracy','f1_macro','f1_weighted','coverage','n_samples')},
                 'runtime_s': round(runtime_s, 2)})
    print(f'FinBERT {ds_name}: acc={m["accuracy"]:.3f}  f1_macro={m["f1_macro"]:.3f}')

In [ ]:
df = pd.DataFrame(rows)
out = os.path.join(cfg['paths']['summary_dir'], 'baselines.csv')
df.to_csv(out, index=False)
print(f'Saved → {out}')
df